In [ ]:
%pip install kagglehub
%pip install tensorflow keras
%pip install scikit-learn

In [ ]:
import kagglehub
import os
import shutil

data_dir = os.path.join(os.getcwd(), "content", "louisiana_images")
key_file = os.path.join(data_dir, "train.csv")

if not os.path.exists(key_file):
    path = kagglehub.dataset_download("rahultp97/louisiana-flood-2016")

    os.makedirs(data_dir, exist_ok=True)

    shutil.copytree(path, data_dir, dirs_exist_ok=True)

    print(f"Dataset copiado de {path} para {data_dir}")
else:
    print(f"Dataset encontrado em {data_dir}")


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv(data_dir + '/train.csv')
test_df = pd.read_csv(data_dir + '/test.csv')

In [ ]:
import tensorflow as tf
import os

image_size = (512, 360)
batch_size = 32

def preprocess_image_py(image_path_tensor, label):
    image_path = image_path_tensor.numpy().decode('utf-8')
    try:
        img = tf.io.read_file(image_path)
        img = tf.image.decode_image(img, channels=3)
        if img.shape.rank == 0:  # Verifica se a imagem está vazia
            raise ValueError(f"imagem vazia")
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32) / 255.0  # Normaliza para o intervalo [0, 1]
        return img, label
    except Exception as e:
        print(f"Erro ao processar a imagem {image_path}: {e}")
        # Retorna uma imagem preta preenchida com zeros como substituta
        return tf.zeros((*image_size, 3), dtype=tf.float32), label

def preprocess_image(image_path, label):
    processed_image, processed_label = tf.py_function(
        preprocess_image_py,
        inp=[image_path, label],
        Tout=[tf.float32, label.dtype]
    )
    processed_image.set_shape([*image_size, 3])
    processed_label.set_shape([])
    return processed_image, processed_label

# --- Criar Dataset de Treino ---
print("Criando dataset de treino...")
train_image_filenames = train_df['Image ID'].values
train_labels = train_df['Flooded'].values

train_image_paths = [os.path.join(data_dir, 'train', fname) for fname in train_image_filenames]

train_ds = tf.data.Dataset.from_tensor_slices((train_image_paths, train_labels))

train_ds = train_ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(buffer_size=len(train_image_paths)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# --- Criar Dataset de Teste ---
print("Criando dataset de teste...")
test_image_filenames = test_df['Image ID'].values
test_labels = test_df['Flooded'].values

test_image_paths = [os.path.join(data_dir, 'test', fname) for fname in test_image_filenames]

test_ds = tf.data.Dataset.from_tensor_slices((test_image_paths, test_labels))
test_ds = test_ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE) # Não embaralha o conjunto de teste

print("Datasets de treino e teste criados com sucesso.")
print(f"Número de amostras de treino: {len(train_image_paths)}")
print(f"Número de amostras de teste: {len(test_image_paths)}")

# Opcional: Verificar um batch do dataset de treino
for images, labels in train_ds.take(1):
    print(f"Formato do batch de imagens de treino: {images.shape}")
    print(f"Formato do batch de labels de treino: {labels.shape}")
    print(f"Tipo de dados do batch de imagens de treino: {images.dtype}")
    print(f"Tipo de dados do batch de labels de treino: {labels.dtype}")
    break

In [ ]:
for imgs, labels in train_ds.take(1):
    print("Valor mínimo:", tf.reduce_min(imgs).numpy())
    print("Valor máximo:", tf.reduce_max(imgs).numpy())

In [ ]:
from keras.applications import MobileNet
from keras.models import Model
from keras.layers import Dense, Flatten, Dropout
from keras.optimizers import Adam
from tensorflow.keras import layers

base_model = MobileNet(
    include_top=False,
    weights='imagenet',
    input_shape=(image_size[0], image_size[1], 3),  # Seu tamanho (512, 360, 3)
    alpha=1.0,
    depth_multiplier=1,
    dropout=0.001
)

# Congela as camadas do modelo base para não serem treinadas
base_model.trainable = False

x = base_model.output

# Camadas para classificação binária
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
output_layer = layers.Dense(1, activation='sigmoid')(x)

# Cria o modelo final
model = Model(inputs=base_model.input, outputs=output_layer)

# Compila o modelo
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Exibe o resumo do modelo
model.summary()

print("Modelo de Transfer Learning criado e compilado com sucesso!")

# Treino do Modelo

In [ ]:
# cria um checkpoint para salvar os pesos do melhor modelo encontrado no treinamento
checkpointer = ModelCheckpoint(filepath='model.weights.best.keras', verbose=1, save_best_only=True)

# treina o modelo
hist = model.fit(train_ds, epochs=100,
          validation_data=test_ds,
          callbacks=[checkpointer],
          verbose=2)

# Resultado
## Matriz de Confusão

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = []
y_pred = []
for images, labels in test_ds.unbatch().batch(1):
    pred = model.predict(images, verbose=0)
    y_true.append(labels.numpy()[0])
    y_pred.append(1 if pred[0][0] > 0.5 else 0)

print("Matriz de confusão:\n", confusion_matrix(y_true, y_pred))
print("\nRelatório por classe:")
print(classification_report(y_true, y_pred, target_names=['Não inundação', 'Inundação']))

## Carregando os pesos do modelo treinado

In [ ]:
model_path = "model.weights.best.keras"   # ou o caminho completo

model = tf.keras.models.load_model(model_path)
model.summary()

## Resultado

In [ ]:
import numpy as np

# Listas para acumular
images_list = []
labels_list = []

for img_batch, lbl_batch in test_ds.unbatch().batch(1):
    images_list.append(img_batch.numpy()[0])
    labels_list.append(lbl_batch.numpy()[0])

# Converte para arrays
x_test = np.array(images_list)
y_test = np.array(labels_list)

print(x_test.shape, y_test.shape)

y_hat_prob = model.predict(x_test)
y_hat = (y_hat_prob > 0.5).astype(int).flatten()

class_names = ['Não inundação', 'Inundação']

fig = plt.figure(figsize=(20, 8))
for i, idx in enumerate(np.random.choice(x_test.shape[0], size=32, replace=False)):
    ax = fig.add_subplot(4, 8, i + 1, xticks=[], yticks=[])
    # Mostra a imagem (os valores estão entre 0 e 1; o imshow entende)
    ax.imshow(np.squeeze(x_test[idx]))
    pred_label = class_names[y_hat[idx]]
    true_label = class_names[y_test[idx]]
    color = "green" if y_hat[idx] == y_test[idx] else "red"
    ax.set_title(f"{pred_label}\n({true_label})", color=color)

fig.subplots_adjust(hspace=0.5)
plt.show()


---
# Atividade 2 – Agente de Monitoramento

Agente reativo que simula múltiplos sensores recebendo imagens de locais diferentes,
classifica cada imagem com o modelo treinado e emite alertas de enchente.

In [ ]:
import os, random, time
import numpy as np

# --- Sensor: representa um drone/câmera em um local ---
class Sensor:
    def __init__(self, nome, local, pasta):
        self.nome  = nome
        self.local = local
        self.imagens = [
            os.path.join(pasta, f) for f in os.listdir(pasta)
            if f.lower().endswith(('.png', '.jpg'))
        ]

    def capturar(self):
        """Devolve o caminho de uma imagem aleatória."""
        return random.choice(self.imagens)

# --- Agente de monitoramento: lê sensores e classifica ---
class AgenteMonitoramento:
    LIMIAR = 0.5

    def __init__(self, modelo, sensores):
        self.modelo   = modelo
        self.sensores = sensores

    def _classificar(self, caminho):
        img = tf.io.read_file(caminho)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32) / 255.0
        t0  = time.perf_counter()
        prob = float(self.modelo.predict(img[None], verbose=0)[0][0])
        ms   = (time.perf_counter() - t0) * 1000
        return prob, ms

    def monitorar(self, ciclos=3):
        log = []
        print(f"Monitoramento iniciado — {len(self.sensores)} sensores, {ciclos} ciclos\n")
        for ciclo in range(1, ciclos + 1):
            print(f"--- Ciclo {ciclo} ---")
            for sensor in self.sensores:
                caminho = sensor.capturar()
                prob, ms = self._classificar(caminho)
                enchente = prob >= self.LIMIAR
                alerta   = "ALERTA" if enchente else "normal"
                print(f"  [{sensor.nome}] {sensor.local}")
                print(f"    {os.path.basename(caminho)} → {'Inundação' if enchente else 'Sem inundação'} "
                      f"({prob:.0%}) [{alerta}] | {ms:.0f} ms")
                log.append({"sensor": sensor.nome, "local": sensor.local,
                             "arquivo": os.path.basename(caminho),
                             "prob": prob, "enchente": enchente, "ms": ms})
            print()

        total    = len(log)
        enchentes = sum(r["enchente"] for r in log)
        print(f"Resumo: {enchentes}/{total} imagens com enchente detectada")
        print(f"Tempo médio de inferência: {np.mean([r['ms'] for r in log]):.0f} ms")
        return log

# --- Configuração e execução ---
sensores = [
    Sensor("DRONE-A", "Baton Rouge – Zona Norte",  os.path.join(data_dir, 'test')),
    Sensor("DRONE-B", "Lafayette – Margem do Rio", os.path.join(data_dir, 'test')),
    Sensor("CAM-01",  "New Orleans – Centro",      os.path.join(data_dir, 'test')),
    Sensor("CAM-02",  "Shreveport – Industrial",   os.path.join(data_dir, 'test')),
]

agente = AgenteMonitoramento(model, sensores)
log    = agente.monitorar(ciclos=3)